In [8]:
taxi_year   = 2024
write_mode  = "overwrite"  

StatementMeta(, 20d99064-7053-4ca4-8290-9300a4aeb821, 10, Finished, Available, Finished, False)

In [9]:
import json
from datetime import datetime, timezone

def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()

run_log = {
    "run_id":  datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
    "phase":   "silver",
    "started": utcnow(),
    "steps":   {},
}

def run_step(name: str, notebook: str, args: dict = None):
    print(f"\n{'─'*60}")
    print(f"  Running : {notebook}")
    print(f"{'─'*60}\n")
    try:
        notebookutils.notebook.run(
            notebook,
            timeout_seconds=7200,
            arguments=args or {},
        )
        run_log["steps"][name] = {"status": "success", "ts": utcnow()}
        print(f"\n  [PASS] {name}")
    except Exception as exc:
        run_log["steps"][name] = {"status": "failed", "error": str(exc), "ts": utcnow()}
        print(f"\n  [FAIL] {name}: {exc}")

StatementMeta(, 20d99064-7053-4ca4-8290-9300a4aeb821, 11, Finished, Available, Finished, False)

In [10]:
run_step(
    name     = "dim_tables",
    notebook = "04_silver_dim_tables",
    args     = {"write_mode": write_mode},
)

StatementMeta(, 20d99064-7053-4ca4-8290-9300a4aeb821, 12, Finished, Available, Finished, False)


────────────────────────────────────────────────────────────
  Running : 04_silver_dim_tables
────────────────────────────────────────────────────────────




  [PASS] dim_tables


In [11]:
run_step(
    name     = "taxi",
    notebook = "01_silver_taxi_transform",
    args     = {"year": taxi_year, "write_mode": write_mode},
)

StatementMeta(, 20d99064-7053-4ca4-8290-9300a4aeb821, 13, Finished, Available, Finished, False)


────────────────────────────────────────────────────────────
  Running : 01_silver_taxi_transform
────────────────────────────────────────────────────────────




  [PASS] taxi


In [12]:
run_step(
    name     = "openaq",
    notebook = "02_silver_openaq_transform",
    args     = {"write_mode": write_mode},
)

StatementMeta(, 20d99064-7053-4ca4-8290-9300a4aeb821, 14, Finished, Available, Finished, False)


────────────────────────────────────────────────────────────
  Running : 02_silver_openaq_transform
────────────────────────────────────────────────────────────




  [PASS] openaq


In [13]:
run_step(
    name     = "gdp_fx",
    notebook = "03_silver_gdp_fx_transform",
    args     = {"write_mode": write_mode},
)

StatementMeta(, 20d99064-7053-4ca4-8290-9300a4aeb821, 15, Finished, Available, Finished, False)


────────────────────────────────────────────────────────────
  Running : 03_silver_gdp_fx_transform
────────────────────────────────────────────────────────────




  [PASS] gdp_fx


In [14]:
run_log["finished"] = utcnow()

successes = sum(1 for v in run_log["steps"].values() if v["status"] == "success")
failures  = sum(1 for v in run_log["steps"].values() if v["status"] == "failed")

print(f"\n{'='*60}")
print(f"  SILVER TRANSFORM COMPLETE")
print(f"  Passed : {successes}/4 steps")
print(f"  Failed : {failures}/4 steps")
print(f"  Run ID : {run_log['run_id']}")
print(f"{'='*60}\n")
print(json.dumps(run_log, indent=2))

import os
log_dir = "/lakehouse/default/Files/_run_logs"
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, f"silver_run_{run_log['run_id']}.json")
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=2)
print(f"\n  [OK] Run log saved: {log_path}")

if failures > 0:
    raise RuntimeError(
        f"Silver transform completed with {failures} failure(s). "
        f"Check run log: {log_path}"
    )

StatementMeta(, 20d99064-7053-4ca4-8290-9300a4aeb821, 16, Finished, Available, Finished, False)


  SILVER TRANSFORM COMPLETE
  Passed : 4/4 steps
  Failed : 0/4 steps
  Run ID : 20260522T091235Z

{
  "run_id": "20260522T091235Z",
  "phase": "silver",
  "started": "2026-05-22T09:12:35.815152+00:00",
  "steps": {
    "dim_tables": {
      "status": "success",
      "ts": "2026-05-22T09:13:10.977936+00:00"
    },
    "taxi": {
      "status": "success",
      "ts": "2026-05-22T09:15:18.125841+00:00"
    },
    "openaq": {
      "status": "success",
      "ts": "2026-05-22T09:18:03.425879+00:00"
    },
    "gdp_fx": {
      "status": "success",
      "ts": "2026-05-22T09:18:51.164897+00:00"
    }
  },
  "finished": "2026-05-22T09:18:52.632697+00:00"
}

  [OK] Run log saved: /lakehouse/default/Files/_run_logs/silver_run_20260522T091235Z.json
